# Data coverage check

**Purpose.** Inspect lake identifiers and the availability of water-quality observations.

**Inputs.** Local CCI Lakes tables under `Datasets/`.

**Outputs.** Coverage summaries used to select lakes for subsequent processing.

> Historical research notebook. Paths assume the repository layout described in `data/README.md`; generated outputs are intentionally not stored in the notebook.


In [ ]:
from pathlib import Path
import os

start_dir = Path.cwd().resolve()
for candidate in (start_dir, *start_dir.parents):
    if (candidate / 'notebooks').is_dir() and (candidate / 'README.md').is_file():
        os.chdir(candidate)
        break
else:
    raise RuntimeError('Run this notebook from inside the cloned repository.')


In [ ]:
import os
import re
import geopandas as gpd

# Paths
project_root = os.getcwd()
lake_file = os.path.abspath(os.path.join(project_root, "Datasets/lakes/CCILakesV202.shp"))
csv_folder = os.path.abspath(os.path.join(project_root, "Datasets/CNR/CHLA/"))

# Step 1: Load and print some sample lake IDs from shapefile
lake_gdf = gpd.read_file(lake_file)
lake_ids_raw = lake_gdf['Lake_ID'].dropna()

print("Sample shapefile LAKE_IDs:")
print(lake_ids_raw.head(10))

# Normalize them as strings of integers
lake_ids = set(lake_ids_raw.astype(int).astype(str))
print(f"\nNormalized LAKE_IDs (shapefile): {sorted(list(lake_ids))[:10]}")

# Step 2: Get lake IDs from filenames
csv_files = os.listdir(csv_folder)
lake_ids_in_folder = set()

for filename in csv_files:
    match = re.match(r"ID(\d+)_", filename)
    if match:
        lake_ids_in_folder.add(match.group(1))

print(f"\nSample lake IDs from filenames: {sorted(list(lake_ids_in_folder))[:10]}")

# Step 3: Normalize both sets
lake_ids_folder_normalized = {str(int(id)) for id in lake_ids_in_folder}

print(f"\nNormalized LAKE_IDs (from folder): {sorted(list(lake_ids_folder_normalized))[:10]}")

# Step 4: Find difference
missing_lakes = lake_ids - lake_ids_folder_normalized

print(f"\nTotal in shapefile: {len(lake_ids)}")
print(f"Total in folder: {len(lake_ids_folder_normalized)}")
print(f"Missing lake IDs: {sorted(list(missing_lakes))}")


In [ ]:
import os
import re
# Set folder paths
chla_folder = "Datasets/CNR/CHLA/"
turb_folder = "Datasets/CNR/turbidity/"

# Regex pattern to extract lake ID
pattern = re.compile(r"ID(\d+)_")

# Helper to extract IDs from filenames in a folder
def extract_ids(folder_path):
    ids = set()
    for filename in os.listdir(folder_path):
        match = pattern.match(filename)
        if match:
            ids.add(match.group(1))
    return ids

# Extract IDs
chla_ids = extract_ids(chla_folder)
turb_ids = extract_ids(turb_folder)

# Find mismatches
only_in_chla = chla_ids - turb_ids
only_in_turb = turb_ids - chla_ids
not_in_both = only_in_chla.union(only_in_turb)

# Output
print(f"IDs only in CHLA: {sorted(only_in_chla)}")
print(f"IDs only in TURB: {sorted(only_in_turb)}")
print(f"IDs NOT in both: {sorted(not_in_both)}")
print(f"\nNumber of lakes with both: {len(chla_ids & turb_ids)}")


In [ ]:
import os
import re

# Set folder paths
chla_folder = "Datasets/CNR/CHLA/"
turb_folder = "Datasets/CNR/turbidity/"

# Regex pattern to extract lake ID
pattern = re.compile(r"ID(\d+)_")

# Helper to extract IDs from filenames in a folder
def extract_ids(folder_path):
    ids = set()
    for filename in os.listdir(folder_path):
        match = pattern.match(filename)
        if match:
            ids.add(match.group(1))
    return ids

# Extract IDs
chla_ids = extract_ids(chla_folder)
turb_ids = extract_ids(turb_folder)

# Find mismatches
only_in_chla = chla_ids - turb_ids
only_in_turb = turb_ids - chla_ids
not_in_both = only_in_chla.union(only_in_turb)

# Output
print(f"IDs only in CHLA: {sorted(only_in_chla)}")
print(f"IDs only in TURB: {sorted(only_in_turb)}")
print(f"IDs NOT in both: {sorted(not_in_both)}")
print(f"\nNumber of lakes with both: {len(chla_ids & turb_ids)}")
